In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag $k$ to repitch the clarinet recording of A#5 (faint red) by $k$ steps
(blue). On the left, the whole note grows or shrinks in time. On the right,
a 5 ms close-up shows its cycles bunching up or spreading out. The two
always move together, because both come from reading the recording faster or
slower. The audio card plays the repitched note.

In [ ]:
# hide
# autorun
K0 = 4                              # up to D6, as in the chapter's figure

clarinet = pq.Audio.from_file("../assets/clarinet.wav").as_mono()
SR = clarinet.sample_rate
X = np.asarray(clarinet.samples).reshape(-1)
X = X / np.abs(X).max()
N = len(X)
T1 = N / SR
F0 = 932.33                         # A#5
MIDI0 = 82                          # A#5 as a MIDI pitch
NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
ZOOM_AT, ZOOM_MS = 1.0, 5.0         # the close-up: 5 ms from 1 s into the recording


def resample(x, ratio):
    """Read x at `ratio` times its original speed, with linear interpolation."""
    p = np.arange(int(len(x) / ratio)) * ratio
    return np.interp(p, np.arange(len(x)), x)


def envelope(x, dt, n_blocks=700):
    """Times and min/max values of x in n_blocks blocks, so peaks survive."""
    step = max(1, len(x) // n_blocks)
    n = len(x) // step * step
    blocks = x[:n].reshape(-1, step)
    t = (np.arange(n // step) + 0.5) * step * dt
    return np.repeat(t, 2), np.column_stack([blocks.min(1), blocks.max(1)]).ravel()


def views(k, X=X, SR=SR, resample=resample, envelope=envelope):
    y = resample(X, 2 ** (k / 12))
    t, e = envelope(y, 1 / SR)
    # the same moment of the recording, ZOOM_AT seconds in, now arrives sooner
    # or later; show ZOOM_MS of output from there
    start = int(ZOOM_AT * SR / 2 ** (k / 12))
    zoom = y[start:start + int(ZOOM_MS / 1000 * SR)]
    return y, t, e, zoom


T_ORIG, E_ORIG = envelope(X, 1 / SR)
Z_ORIG = X[int(ZOOM_AT * SR):int(ZOOM_AT * SR) + int(ZOOM_MS / 1000 * SR)]
TZ = np.arange(len(Z_ORIG)) / SR * 1000


def figure():
    y, t, e, zoom = views(K0)
    fig = make_subplots(rows=1, cols=2, column_widths=[0.66, 0.34],
                        horizontal_spacing=0.09)
    fig.add_scatter(x=T_ORIG, y=E_ORIG, mode="lines", opacity=0.35,
                    line=dict(color=RED, width=1), row=1, col=1)
    fig.add_scatter(x=t, y=e, mode="lines", line=dict(color=BLUE, width=1),
                    row=1, col=1)
    fig.add_scatter(x=TZ, y=Z_ORIG, mode="lines", opacity=0.45,
                    line=dict(color=RED, width=1.4, dash="dash"), row=1, col=2)
    fig.add_scatter(x=TZ[:len(zoom)], y=zoom, mode="lines",
                    line=dict(color=BLUE, width=2), row=1, col=2)
    fig.update_xaxes(range=[0, 2 * T1], title_text="Time (s)", fixedrange=True,
                     row=1, col=1)
    fig.update_yaxes(range=[-1.1, 1.1], title_text="Amplitude", fixedrange=True,
                     row=1, col=1)
    fig.update_xaxes(range=[0, ZOOM_MS], title_text="Close-up (ms)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[-1.1, 1.1], showticklabels=False, fixedrange=True,
                     row=1, col=2)
    return fig


def controls(fig):
    k = widgets.IntSlider(description="Steps $k$", min=-12, max=12, value=K0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(k, views=views, NAMES=NAMES, MIDI0=MIDI0, F0=F0, T1=T1,
               readout=readout):
        y, t, e, zoom = views(k)
        with fig.batch_update():
            fig.data[1].x, fig.data[1].y = t, e
            fig.data[3].x, fig.data[3].y = fig.data[2].x[:len(zoom)], zoom
        m = MIDI0 + k
        name = f"{NAMES[m % 12]}{m // 12 - 1}"
        r = 2 ** (k / 12)
        readout.value = (
            f"<span style='font-size:0.9em'><i>T</i><sub>1</sub> / "
            f"<i>T</i><sub>2</sub> = 2<sup>{k}/12</sup> = {r:.3f} &nbsp;·&nbsp; "
            f"A#5 &rarr; {name} ({F0:.0f} &rarr; {F0 * r:.0f} Hz) &nbsp;·&nbsp; "
            f"duration {T1:.2f} &rarr; {T1 / r:.2f} s</span>")

    widgets.interactive_output(update, {"k": k})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(views=views):
        y = 0.5 * views(k.value)[0]
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (k,):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([k, readout, out, gate])

icm_plotly.show(figure, controls)